# SystemDesignExpertLLM — SFT (QLoRA) on Kaggle

Fine-tunes `unsloth/Qwen2.5-7B-Instruct` (4-bit QLoRA) on the corpus-grounded SFT dataset.

**Runtime:** Kaggle *GPU* (single P100 16GB preferred, or one T4).

**Data:** upload `data/generated/sft.jsonl` as a Kaggle Dataset and attach it; set `SFT_PATH` below.

Checkpoints are saved to `/kaggle/working/adapters/sft` every `save_steps` so a 12h session cutoff can be resumed.

In [ ]:
%%capture
# Unsloth pins compatible torch/transformers/trl/peft/bitsandbytes wheels.
!pip install -q "unsloth[kaggle-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps trl peft accelerate bitsandbytes datasets

In [ ]:
import os

# --- config (mirror of configs/sft.yaml) ---
BASE_MODEL   = "unsloth/Qwen2.5-7B-Instruct"
MAX_SEQ_LEN  = 4096
SFT_PATH     = "/kaggle/input/sdx-dataset/sft.jsonl"   # <-- point at your attached dataset
# After a 12h session cutoff: save /kaggle/working/adapters/sft as a new Kaggle dataset,
# attach it to a fresh notebook session, and point this at its input path. Leave empty
# for a first run; the training cell copies it into OUT_ADAPTER and resumes when set.
RESUME_FROM  = ""   # e.g. "/kaggle/input/sdx-sft-checkpoint"
OUT_ADAPTER  = "/kaggle/working/adapters/sft"
OUT_MERGED   = "/kaggle/working/merged/sft-fp16"
EPOCHS       = 3
BATCH        = 2
GRAD_ACCUM   = 8       # effective batch 16
LR           = 2e-4
SAVE_STEPS   = 100
SEED         = 3407
os.makedirs(OUT_ADAPTER, exist_ok=True)

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = BASE_MODEL,
    max_seq_length = MAX_SEQ_LEN,
    dtype = None,            # auto (bf16 if supported else fp16)
    load_in_4bit = True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r = 32,
    lora_alpha = 32,
    lora_dropout = 0.05,
    target_modules = ["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    use_gradient_checkpointing = "unsloth",
    random_state = SEED,
)

In [ ]:
from datasets import load_dataset

SYSTEM_PROMPT = (
    "You are a distinguished software architect. Given a real requirement, you produce "
    "concrete, production-grade system design guidance with explicit technology choices "
    "and rigorous tradeoff analysis."
)

def to_chat(ex):
    msgs = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": ex["instruction"]},
        {"role": "assistant", "content": ex["output"]},
    ]
    return {"text": tokenizer.apply_chat_template(msgs, tokenize=False, add_generation_prompt=False)}

ds = load_dataset("json", data_files=SFT_PATH, split="train").map(to_chat)
split = ds.train_test_split(test_size=0.05, seed=SEED)
train_ds, eval_ds = split["train"], split["test"]
print(train_ds, eval_ds, train_ds[0]["text"][:400])

In [ ]:
from trl import SFTTrainer, SFTConfig

import shutil

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = train_ds,
    eval_dataset = eval_ds,
    dataset_text_field = "text",
    max_seq_length = MAX_SEQ_LEN,
    packing = True,
    args = SFTConfig(
        per_device_train_batch_size = BATCH,
        per_device_eval_batch_size = BATCH,
        gradient_accumulation_steps = GRAD_ACCUM,
        warmup_ratio = 0.05,
        num_train_epochs = EPOCHS,
        learning_rate = LR,
        lr_scheduler_type = "cosine",
        weight_decay = 0.01,
        logging_steps = 10,
        eval_strategy = "steps",
        eval_steps = SAVE_STEPS,
        save_steps = SAVE_STEPS,
        save_total_limit = 3,
        optim = "adamw_8bit",
        seed = SEED,
        output_dir = OUT_ADAPTER,
        report_to = "none",
    ),
)

# Resume after a 12h session cutoff: copy the saved checkpoint dataset into OUT_ADAPTER
# (Kaggle /kaggle/input is read-only, Trainer needs a writable output_dir) and resume.
_resuming = bool(RESUME_FROM) and os.path.isdir(RESUME_FROM)
if _resuming:
    shutil.copytree(RESUME_FROM, OUT_ADAPTER, dirs_exist_ok=True)
    print(f"resuming from checkpoint copied out of {RESUME_FROM}")
trainer.train(resume_from_checkpoint=_resuming)

In [ ]:
# Save the LoRA adapter and a merged fp16 model (for GGUF/HF export).
model.save_pretrained(OUT_ADAPTER); tokenizer.save_pretrained(OUT_ADAPTER)
model.save_pretrained_merged(OUT_MERGED, tokenizer, save_method="merged_16bit")
print("saved adapter ->", OUT_ADAPTER, "| merged ->", OUT_MERGED)

In [ ]:
# Quick sanity generation
FastLanguageModel.for_inference(model)
msgs = [
    {"role": "system", "content": SYSTEM_PROMPT},
    {"role": "user", "content": "Design a multi-region inventory service that keeps stock counts strongly consistent for 5M users."},
]
inputs = tokenizer.apply_chat_template(msgs, tokenize=True, add_generation_prompt=True, return_tensors="pt").to("cuda")
out = model.generate(input_ids=inputs, max_new_tokens=800, temperature=0.7)
print(tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True))